# Maximum Likelihood Estimation

Companion notebook for the [Maximum Likelihood Estimation](https://ml-viz.vercel.app/courses/probability-statistics/03-maximum-likelihood-estimation) lesson.

We'll visualize likelihood functions, derive MLE analytically, and connect MLE to cross-entropy loss.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#30344a', 'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0', 'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
})

## Gaussian MLE: grid search vs. closed form

We derived in the lesson that for $x_i \sim \mathcal{N}(\mu, \sigma^2)$ the log-likelihood is

$$\ell(\mu,\sigma^2) = -\frac{n}{2}\ln(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_i (x_i-\mu)^2,$$

and setting the derivatives to zero gives the closed forms

$$\hat\mu = \frac1n\sum_i x_i = \bar x, \qquad \hat\sigma^2 = \frac1n\sum_i (x_i-\bar x)^2.$$

Below we **scan a grid** of candidate $\mu$, take the argmax of $\ell$, and check it lands on $\bar x$.

In [ ]:
rng = np.random.default_rng(42)
true_mu, true_sigma = 3.0, 1.5
data = rng.normal(true_mu, true_sigma, size=200)
n = data.size

# --- Our own Gaussian log-likelihood (no scipy) -------------------------------
def gaussian_loglik(mu, var, data):
    # ell = -n/2 * ln(2*pi*var) - 1/(2*var) * sum((x - mu)^2)
    return -0.5 * n * np.log(2 * np.pi * var) - np.sum((data - mu) ** 2) / (2 * var)

# --- Closed-form MLE (from the lesson derivation) -----------------------------
mle_mu = data.mean()
mle_var = np.mean((data - mle_mu) ** 2)   # divisor n -> biased MLE variance

# --- Grid search over mu (variance fixed at its MLE) --------------------------
mu_grid = np.linspace(true_mu - 3, true_mu + 3, 6001)
ll_mu = np.array([gaussian_loglik(mu, mle_var, data) for mu in mu_grid])
mu_argmax = mu_grid[ll_mu.argmax()]

# --- Grid search over var (mu fixed at its MLE) -------------------------------
var_grid = np.linspace(0.5, 6.0, 6001)
ll_var = np.array([gaussian_loglik(mle_mu, v, data) for v in var_grid])
var_argmax = var_grid[ll_var.argmax()]

print("Gaussian MLE for the MEAN")
print("  grid argmax mu  = {:.4f}".format(mu_argmax))
print("  closed-form xbar= {:.4f}".format(mle_mu))
print("  true mu         = {:.4f}".format(true_mu))
print()
print("Gaussian MLE for the VARIANCE")
print("  grid argmax var = {:.4f}".format(var_argmax))
print("  closed-form 1/n = {:.4f}".format(mle_var))
print("  unbiased 1/(n-1)= {:.4f}".format(data.var(ddof=1)))
print("  true var        = {:.4f}".format(true_sigma ** 2))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ax = axes[0]
ax.plot(mu_grid, ll_mu, color='#6366f1', lw=2)
ax.axvline(mu_argmax, color='#f97316', lw=2, ls='--',
           label='grid argmax = {:.3f}'.format(mu_argmax))
ax.axvline(true_mu, color='#2dd4bf', lw=2, ls=':', label='true mu = {}'.format(true_mu))
ax.set_xlabel('mu'); ax.set_ylabel('log-likelihood')
ax.set_title('ell(mu | data), var fixed'); ax.legend(); ax.grid(True, alpha=0.2)

ax = axes[1]
ax.plot(var_grid, ll_var, color='#6366f1', lw=2)
ax.axvline(var_argmax, color='#f97316', lw=2, ls='--',
           label='grid argmax = {:.3f}'.format(var_argmax))
ax.axvline(true_sigma ** 2, color='#2dd4bf', lw=2, ls=':',
           label='true var = {:.2f}'.format(true_sigma ** 2))
ax.set_xlabel('sigma^2'); ax.set_ylabel('log-likelihood')
ax.set_title('ell(sigma^2 | data), mu fixed'); ax.legend(); ax.grid(True, alpha=0.2)

ax = axes[2]
x_plot = np.linspace(data.min() - 1, data.max() + 1, 300)
ax.hist(data, bins=20, density=True, alpha=0.6, color='#6366f1', label='data')
mle_pdf = (1 / np.sqrt(2 * np.pi * mle_var)) * np.exp(-(x_plot - mle_mu) ** 2 / (2 * mle_var))
true_pdf = (1 / np.sqrt(2 * np.pi * true_sigma ** 2)) * np.exp(-(x_plot - true_mu) ** 2 / (2 * true_sigma ** 2))
ax.plot(x_plot, mle_pdf, color='#f97316', lw=2,
        label='MLE fit N({:.2f}, {:.2f})'.format(mle_mu, mle_var))
ax.plot(x_plot, true_pdf, color='#2dd4bf', lw=2, ls=':',
        label='true N({}, {:.2f})'.format(true_mu, true_sigma ** 2))
ax.set_xlabel('x'); ax.set_ylabel('density')
ax.set_title('MLE fit vs true distribution'); ax.legend(); ax.grid(True, alpha=0.2)

plt.tight_layout(); plt.show()

## Bernoulli MLE: grid search vs. closed form

For $x_i \sim \text{Bernoulli}(p)$ the log-likelihood is $\ell(p) = k\ln p + (n-k)\ln(1-p)$ with $k=\sum_i x_i$.
Setting $d\ell/dp = 0$ gives $\hat p = k/n$ — the empirical success rate. We confirm the grid argmax matches.

In [ ]:
rng = np.random.default_rng(7)
true_p = 0.7
coins = rng.binomial(1, true_p, size=200)   # 0/1 outcomes
k = coins.sum()
m = coins.size

def bernoulli_loglik(p, k, m):
    eps = 1e-12
    return k * np.log(p + eps) + (m - k) * np.log(1 - p + eps)

# Grid search over p
p_grid = np.linspace(0.001, 0.999, 9999)
ll_p = np.array([bernoulli_loglik(p, k, m) for p in p_grid])
p_argmax = p_grid[ll_p.argmax()]

p_closed = k / m   # = coins.mean()

print("Bernoulli MLE")
print("  grid argmax p    = {:.4f}".format(p_argmax))
print("  closed-form k/n  = {:.4f}".format(p_closed))
print("  true p           = {:.4f}".format(true_p))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(p_grid, ll_p, color='#6366f1', lw=2)
ax.axvline(p_argmax, color='#f97316', lw=2, ls='--',
           label='grid argmax = {:.3f}'.format(p_argmax))
ax.axvline(true_p, color='#2dd4bf', lw=2, ls=':', label='true p = {}'.format(true_p))
ax.set_xlabel('p'); ax.set_ylabel('log-likelihood')
ax.set_title('Bernoulli ell(p | {} successes / {} trials)'.format(k, m))
ax.legend(); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

## MLE = minimizing NLL = cross-entropy

The per-example negative log-likelihood of a Bernoulli model is exactly binary cross-entropy:

$$\frac1n\,\text{NLL} = -\frac1n\sum_i \big[y_i\ln\hat y_i + (1-y_i)\ln(1-\hat y_i)\big] = \mathcal{L}_{CE}.$$

So fitting logistic regression by minimizing cross-entropy *is* MLE. We minimize it by plain
gradient descent (no scipy optimizer) and recover the data-generating parameters.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Binary classification: one feature, logistic model
rng = np.random.default_rng(1)
nobs = 400
X = rng.normal(0, 1, nobs)
true_w, true_b = 2.0, -0.5
p_true = sigmoid(true_w * X + true_b)
y = rng.binomial(1, p_true)

def cross_entropy(w, b, X, y):
    p = sigmoid(w * X + b)
    eps = 1e-12
    return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

# --- Minimize cross-entropy (= maximize log-likelihood) by gradient descent ---
# d/dw CE = mean((p - y) * X) ;  d/db CE = mean(p - y)
w, b = 0.0, 0.0
lr = 0.5
for step in range(5000):
    p = sigmoid(w * X + b)
    grad_w = np.mean((p - y) * X)
    grad_b = np.mean(p - y)
    w -= lr * grad_w
    b -= lr * grad_b

print("True parameters:  w = {}, b = {}".format(true_w, true_b))
print("MLE parameters:   w = {:.4f}, b = {:.4f}".format(w, b))
print("Final cross-entropy (= NLL/n): {:.4f}".format(cross_entropy(w, b, X, y)))

fig, ax = plt.subplots(figsize=(9, 5))
x_plot = np.linspace(-3.5, 3.5, 300)
ax.scatter(X[y == 0], y[y == 0], color='#6366f1', alpha=0.4, s=25, label='y=0')
ax.scatter(X[y == 1], y[y == 1], color='#f97316', alpha=0.4, s=25, label='y=1')
ax.plot(x_plot, sigmoid(true_w * x_plot + true_b), color='#2dd4bf', lw=2, ls=':',
        label='true p(y=1|x)')
ax.plot(x_plot, sigmoid(w * x_plot + b), color='#f59e0b', lw=2.5, label='MLE fit')
ax.set_xlabel('x'); ax.set_ylabel('P(y=1 | x)')
ax.set_title('Logistic regression via MLE (= cross-entropy minimization)')
ax.legend(); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Gaussian negative log-likelihood

Taking $-\log$ of the Gaussian likelihood of $n$ i.i.d. points turns the product into a sum:

$$\text{NLL}(\mu, \sigma) = \frac{n}{2}\log(2\pi\sigma^2) + \frac{\sum_i (x_i - \mu)^2}{2\sigma^2}$$

Implement it. The checks confirm it equals $-\sum_i \log p(x_i)$ computed directly, and that the **sample mean** beats nearby values of $\mu$ — the closed-form MLE you derived above.

In [ ]:
def gaussian_nll(data, mu, sigma):
    """Negative log-likelihood of the data under N(mu, sigma^2)."""
    data = np.asarray(data, dtype=float)
    n = len(data)

    # TODO(you): the constant term n/2 * log(2 pi sigma^2)
    const = ...

    # TODO(you): the data term sum((x - mu)^2) / (2 sigma^2)
    fit = ...

    return const + fit

In [ ]:
# Checks — run me
rng = np.random.default_rng(0)
data = rng.normal(1.7, 0.9, size=300)
mu_hat = data.mean()

assert gaussian_nll(data, mu_hat, 0.9) < gaussian_nll(data, mu_hat + 0.1, 0.9), "sample mean beats mu + 0.1"
assert gaussian_nll(data, mu_hat, 0.9) < gaussian_nll(data, mu_hat - 0.1, 0.9), "sample mean beats mu - 0.1"

def gaussian_pdf(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))

direct = -np.sum(np.log(gaussian_pdf(data, 0.5, 1.2)))
assert abs(gaussian_nll(data, 0.5, 1.2) - direct) < 1e-8, "must equal -sum(log pdf)"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def gaussian_nll(data, mu, sigma):
    data = np.asarray(data, dtype=float)
    n = len(data)
    const = n / 2 * np.log(2 * np.pi * sigma ** 2)
    fit = np.sum((data - mu) ** 2) / (2 * sigma ** 2)
    return const + fit
```

</details>

### Exercise 2 — Poisson MLE by grid search

For count data $x_1, \dots, x_n \sim \text{Poisson}(\lambda)$, dropping the $\lambda$-free terms leaves

$$\text{NLL}(\lambda) = n\lambda - \left(\textstyle\sum_i x_i\right) \log \lambda$$

Implement it and minimize over a grid, exactly like the Gaussian and Bernoulli grid searches above. Calculus says the answer is the **sample mean** — your grid search should land there.

In [ ]:
def poisson_nll(data, lam):
    """Poisson negative log-likelihood, constants dropped."""
    data = np.asarray(data, dtype=float)

    # TODO(you): n * lam - sum(data) * log(lam)
    return ...


def poisson_mle_grid(data, grid):
    """The lambda in `grid` with the smallest NLL."""
    # TODO(you): evaluate poisson_nll for each lambda and pick the argmin
    return ...

In [ ]:
# Checks — run me
grid = np.linspace(0.1, 10, 991)

data = np.array([2, 3, 4, 3, 5, 1, 3, 3])
assert abs(poisson_mle_grid(data, grid) - data.mean()) < 0.01, "Poisson MLE is the sample mean"

data2 = np.array([7, 9, 8, 8])
assert abs(poisson_mle_grid(data2, grid) - 8.0) < 0.01, "mean 8 -> lambda_hat ~ 8"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def poisson_nll(data, lam):
    data = np.asarray(data, dtype=float)
    return len(data) * lam - np.sum(data) * np.log(lam)


def poisson_mle_grid(data, grid):
    nlls = [poisson_nll(data, lam) for lam in grid]
    return grid[int(np.argmin(nlls))]
```

</details>